In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

pinecone_api_key = os.getenv("PINECONE_API_KEY")



In [5]:
!pip install pinecone

   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   -------- ------------------------------- 0.5/2.6 MB 4.6 MB/s eta 0:00:01
   ---------------------------- ----------- 1.8/2.6 MB 5.6 MB/s eta 0:00:01
   ---------------------------------------- 2.6/2.6 MB 5.6 MB/s  0:00:00

   ---------------- ----------------------- 2/5 [hpack]
   ---------------- ----------------------- 2/5 [hpack]
   ------------------------ --------------- 3/5 [h2]
   -------------------------------- ------- 4/5 [pinecone]
   -------------------------------- ------- 4/5 [pinecone]
   -------------------------------- ------- 4/5 [pinecone]
   -------------------------------- ------- 4/5 [pinecone]
   -------------------------------- ------- 4/5 [pinecone]
   -------------------------------- ------- 4/5 [pinecone]
   -------------------------------- ------- 4/5 [pinecone]
   -------------------------------- ------- 4/5 [pinecone]
   -------------------------------- ------- 4/5 [pinecone]
   ------

In [17]:
from pinecone import Pinecone, ServerlessSpec

In [7]:
import os

In [14]:
pc = Pinecone(api_key=os.getenv('PINECONE_API_KEY'))

In [16]:
pc.list_indexes()

IndexList([<name='test', dim=1024, ready=True>])

In [18]:
pc.create_index(
    name="my-index",
    dimension=3,
    metric="cosine",
    spec = ServerlessSpec(
        cloud="aws",
        region="us-east-1",
    )
)

IndexModel(name='my-index', metric='cosine', status=IndexStatus(ready=True, state='Ready'), spec=IndexSpec(serverless=ServerlessSpecInfo(cloud='aws', region='us-east-1', read_capacity={'mode': 'OnDemand', 'status': {'state': 'Ready', 'current_shards': None, 'current_replicas': None}}, source_collection=None, schema=None), pod=None, byoc=None), host='https://my-index-v29tsf3.svc.aped-4627-b74a.pinecone.io', private_host=None, vector_type='dense', dimension=3, deletion_protection='disabled', tags=None, embed=None, created_at=None)

In [20]:
pc.list_indexes()

IndexList([<name='my-index', dim=3, ready=True>])

In [24]:
data = [
    {"id": "vec1", "values": [0.1, 0.2, 0.3]},
    {"id": "vec2", "values": [0.4, 0.5, 0.6]},
    {"id": "vec3", "values": [0.7, 0.8, 0.9]} # only floats allowed
]
idx = pc.Index("my-index")

idx.upsert(vectors=data)

UpsertResponse(upserted_count=3)

In [25]:
# update operation of a vector
idx.update(id="vec1", values=[0.11, 0.22, 0.33])

UpdateResponse(matched_records=None, response_info=ResponseInfo(raw_headers={'date': 'Sun, 28 Jun 2026 08:38:56 GMT', 'content-type': 'application/json', 'content-length': '2', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '2', 'x-pinecone-request-logical-size': '16', 'x-pinecone-request-latency-ms': '167', 'x-envoy-upstream-service-time': '171', 'x-pinecone-response-duration-ms': '172', 'grpc-status': '0', 'server': 'envoy'}))

In [30]:
# query vector
res = idx.query(
    vector=[0.0, 0.0, 0.0],
    top_k=2
)
res

QueryResponse(matches=[ScoredVector(id='vec2', score=0.0, values=[]), ScoredVector(id='vec1', score=0.0, values=[])], namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Sun, 28 Jun 2026 08:42:46 GMT', 'content-type': 'application/json', 'content-length': '137', 'connection': 'keep-alive', 'x-pinecone-max-indexed-lsn': '2', 'x-pinecone-request-latency-ms': '5', 'x-envoy-upstream-service-time': '6', 'x-pinecone-response-duration-ms': '7', 'grpc-status': '0', 'server': 'envoy'}))

In [31]:
# query vector
res = idx.query(
    vector=[0.0, 0.0, 0.0],
    top_k=2,
    include_values=True
)
res

QueryResponse(matches=[ScoredVector(id='vec2', score=0.0, values=[0.4, 0.5, 0.6]), ScoredVector(id='vec1', score=0.0, values=[0.11, 0.22, 0.33])], namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Sun, 28 Jun 2026 08:45:02 GMT', 'content-type': 'application/json', 'content-length': '162', 'connection': 'keep-alive', 'x-pinecone-max-indexed-lsn': '2', 'x-pinecone-request-latency-ms': '44', 'x-envoy-upstream-service-time': '45', 'x-pinecone-response-duration-ms': '46', 'grpc-status': '0', 'server': 'envoy'}))

In [33]:
# fetch vectors by id 
res = idx.fetch(ids=["vec1", "vec2"])
res

FetchResponse(vectors={'vec2': Vector(id='vec2', values=[0.4, 0.5, 0.6], sparse_values=None), 'vec1': Vector(id='vec1', values=[0.11, 0.22, 0.33], sparse_values=None)}, namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Sun, 28 Jun 2026 08:45:39 GMT', 'content-type': 'application/json', 'content-length': '143', 'connection': 'keep-alive', 'x-pinecone-request-latency-ms': '39', 'x-envoy-upstream-service-time': '40', 'x-pinecone-response-duration-ms': '41', 'grpc-status': '0', 'server': 'envoy'}))

In [34]:
# delete a vector by id 
idx.delete(ids=["vec1"])